[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/ETL_ELT_Patterns.ipynb)

# ETL/ELT Patterns — Hands-On

**Full refresh vs incremental vs MERGE. Build a pipeline with a dead letter queue. All runnable locally — no cloud project needed.**

Playbook: [ETL/ELT Patterns](../../playbooks/data/pipelines/etl-elt/README.md)

---

## What You Will Do

1. **ETL vs ELT** — see the difference with the same data
2. **Full refresh** — drop and reload (simple but slow)
3. **Incremental load** — watermark-based (only new records)
4. **MERGE / Upsert** — handle updated records without duplicates
5. **Dead Letter Queue** — quarantine bad records instead of dropping them
6. **Idempotency** — prove the pipeline is safe to re-run

**Time:** ~20 minutes

**Prerequisites:** Basic Python. No GCP/AWS account needed.

---

## 1. Setup

We use plain Python + pandas to keep it simple. No Spark, no cloud. The patterns are the same regardless of the tool.

In [ ]:
import pandas as pd
from datetime import datetime, timedelta
import json

print("Setup complete")

---

## 2. Create Source Data

Simulate a call center database with 10 call records. This is the "source system" — the application database that our pipeline reads from.

In [ ]:
# WHY: This simulates the source database (e.g., Cloud SQL, RDS).
# In production, you'd read from the actual database or from files in GCS/S3.

source_data = pd.DataFrame({
    "call_id": [f"C-{i:03d}" for i in range(1, 11)],
    "customer_id": [f"CUST-{i:03d}" for i in range(101, 111)],
    "status": ["resolved", "resolved", "missed", "resolved", "voicemail",
               "in-progress", "resolved", "missed", "resolved", "in-progress"],
    "duration": [340, 120, 0, 480, 60, 0, 200, 0, 150, 0],
    "campaign_id": ["CAMP-A", "CAMP-A", "CAMP-B", "CAMP-B", "CAMP-A",
                    "CAMP-C", "CAMP-C", "CAMP-A", "CAMP-B", "CAMP-C"],
    "created_at": pd.date_range("2026-04-13 09:00", periods=10, freq="30min"),
    "updated_at": pd.date_range("2026-04-13 09:00", periods=10, freq="30min"),
})

print(f"Source: {len(source_data)} records")
source_data

**You Should See:** 10 records. Note C-006 and C-010 are `in-progress` — they will be updated later.

---

## 3. ETL vs ELT — See the Difference

Same data, two approaches. The difference is WHERE the transformation happens.

In [ ]:
# === ETL: Extract → Transform → Load ===
# WHY: Transform BEFORE loading into the warehouse.
# The warehouse only sees clean data.

# Step 1: Extract (read from source)
raw = source_data.copy()

# Step 2: Transform OUTSIDE the warehouse (Python/Spark)
# - Add a computed column
# - Filter out missed calls
# - Standardize status to uppercase
transformed = (
    raw
    .assign(status_upper=raw["status"].str.upper())
    .query("status != 'missed'")
    .assign(minutes=lambda df: df["duration"] / 60)
)

# Step 3: Load (write to warehouse — only clean data arrives)
warehouse_etl = transformed.copy()

print(f"ETL result: {len(warehouse_etl)} records (missed calls filtered out before loading)")
warehouse_etl[["call_id", "status", "status_upper", "duration", "minutes"]].head()

In [ ]:
# === ELT: Extract → Load → Transform ===
# WHY: Load RAW data into the warehouse first.
# Transform INSIDE the warehouse using SQL.
# The warehouse keeps the raw data AND the clean version.

# Step 1: Extract (read from source)
raw = source_data.copy()

# Step 2: Load RAW into warehouse (Bronze layer — no changes)
warehouse_bronze = raw.copy()
print(f"Bronze: {len(warehouse_bronze)} records (everything, unchanged)")

# Step 3: Transform INSIDE the warehouse (Silver layer — this would be SQL)
# In BigQuery, this is: CREATE TABLE silver.calls AS SELECT ... FROM bronze.calls WHERE ...
warehouse_silver = (
    warehouse_bronze
    .assign(status_upper=warehouse_bronze["status"].str.upper())
    .query("status != 'missed'")
    .assign(minutes=lambda df: df["duration"] / 60)
)

print(f"Silver: {len(warehouse_silver)} records (transformed inside warehouse)")
print(f"\nThe difference: ELT keeps Bronze (raw) in the warehouse. ETL doesn't.")
print(f"If your transform has a bug, ELT lets you re-run from Bronze. ETL must re-extract from source.")

**Key takeaway:** Same result, different location for the transform.

| | ETL | ELT |
|---|---|---|
| Raw data in warehouse? | No | Yes (Bronze) |
| Can re-run transform without re-extracting? | No | Yes |
| Your GCP pipeline? | | This one |

**Your GCP pipeline is ELT:** Load CSVs/JSONs into BigQuery Bronze, transform with SQL into Silver/Gold.

---

## 4. Pattern: Full Refresh

Drop everything. Reload everything. Every single run.

### Where Does Silver Actually Live?

This is the question that trips people up. Silver is a **concept** (cleaned data), not a specific technology. Where it lives depends on the pattern:

| Pattern | Where Silver Lives | Example |
|---|---|---|
| **ELT** (your GCP pipeline) | BigQuery `silver` dataset | `silver.calls` is a BigQuery table |
| **ETL** (Spark-based) | GCS folder | `gs://bucket/silver/calls/` is Parquet files |
| **Hybrid** (Delta + BigLake) | GCS Delta table, BigQuery reads it | `gs://bucket/silver/calls_delta/` with `_delta_log/` |

**When someone says "write to Silver" — ask: Silver in BigQuery or Silver in GCS?**

The answer depends on the pattern. The concept is the same — cleaned, validated data. The location is different.

In the code above:
- **ETL**: `transformed` (the clean DataFrame) was built OUTSIDE the warehouse, then loaded in
- **ELT**: `warehouse_bronze` was loaded RAW, then `warehouse_silver` was built INSIDE the warehouse

Same result. Different place for the transform. Different place for Silver.

In [ ]:
# WHY: Full refresh is the simplest pattern. Works great for small tables.
# Breaks when data gets large (reloading 10M rows nightly = slow).

def full_refresh(source, target_name):
    """Drop and reload everything."""
    # In BigQuery this would be: TRUNCATE TABLE target; INSERT INTO target SELECT * FROM source;
    target = source.copy()
    print(f"Full refresh: loaded {len(target)} records into {target_name}")
    return target

# Run 1: Load everything
warehouse = full_refresh(source_data, "silver.calls")

# Run 2: Load everything AGAIN (same result — but scanned all 10 records unnecessarily)
warehouse = full_refresh(source_data, "silver.calls")

print(f"\nProblem: Both runs scanned all {len(source_data)} records.")
print(f"At 10 records this is fine. At 10 million, Run 2 wastes hours.")

---

## 5. Pattern: Incremental Load (Watermark)

Only load records that changed since the last run. Track position using a watermark (bookmark).

In [ ]:
# WHY: Incremental avoids re-reading the entire source.
# The watermark answers: "What was the last record I processed?"

# --- Simulate the watermark table ---
watermarks = {"calls": pd.Timestamp("1970-01-01")}  # start from the beginning

def incremental_load(source, watermarks, table_name):
    """Load only records newer than the watermark."""
    watermark = watermarks[table_name]
    
    # Only get records updated AFTER the watermark
    new_records = source[source["updated_at"] > watermark]
    
    # Update the watermark to the latest record we just processed
    if len(new_records) > 0:
        watermarks[table_name] = new_records["updated_at"].max()
    
    print(f"Incremental: {len(new_records)} new records (watermark was {watermark})")
    return new_records

# Run 1: First run — watermark is 1970, so everything is "new"
batch1 = incremental_load(source_data, watermarks, "calls")
print(f"Watermark after Run 1: {watermarks['calls']}")
print()

In [ ]:
# Run 2: No new data — watermark hasn't moved
batch2 = incremental_load(source_data, watermarks, "calls")
print(f"Watermark after Run 2: {watermarks['calls']}")
print("\nZero records processed. The pipeline ran in milliseconds instead of scanning everything.")

In [ ]:
# Run 3: Simulate a new call arriving in the source
new_call = pd.DataFrame({
    "call_id": ["C-011"],
    "customer_id": ["CUST-111"],
    "status": ["resolved"],
    "duration": [200],
    "campaign_id": ["CAMP-A"],
    "created_at": [pd.Timestamp("2026-04-13 15:00")],
    "updated_at": [pd.Timestamp("2026-04-13 15:03")],
})

source_with_new = pd.concat([source_data, new_call], ignore_index=True)

batch3 = incremental_load(source_with_new, watermarks, "calls")
print(f"\nOnly the new record was loaded:")
batch3[["call_id", "status", "duration"]]

**You Should See:** Run 1 = 10 records. Run 2 = 0 records. Run 3 = 1 record. That is the power of incremental — process only what changed.

---

## 6. Pattern: MERGE / Upsert

Incremental handles NEW records. But what about UPDATED records? Call C-006 was `in-progress` — it is now `resolved`. If you INSERT it, you get a duplicate. You need MERGE.

In [ ]:
# WHY: MERGE = INSERT if new, UPDATE if exists.
# This is the most common pattern in production pipelines.

# Start with the current warehouse state (all 10 records)
warehouse = source_data.copy()
print(f"Warehouse before MERGE: {len(warehouse)} records")
print(f"C-006 status: {warehouse[warehouse['call_id'] == 'C-006']['status'].values[0]}")
print(f"C-010 status: {warehouse[warehouse['call_id'] == 'C-010']['status'].values[0]}")

In [ ]:
# Incoming batch: 2 updates + 1 new record
incoming = pd.DataFrame({
    "call_id": ["C-006", "C-010", "C-012"],
    "customer_id": ["CUST-106", "CUST-110", "CUST-112"],
    "status": ["resolved", "resolved", "in-progress"],  # C-006 and C-010 now resolved
    "duration": [480, 320, 0],
    "campaign_id": ["CAMP-C", "CAMP-C", "CAMP-B"],
    "created_at": [pd.Timestamp("2026-04-13 11:30"), pd.Timestamp("2026-04-13 13:30"),
                   pd.Timestamp("2026-04-13 15:00")],
    "updated_at": [pd.Timestamp("2026-04-13 11:38"), pd.Timestamp("2026-04-13 13:35"),
                   pd.Timestamp("2026-04-13 15:00")],
})

print("Incoming batch:")
incoming[["call_id", "status", "duration"]]

In [ ]:
def merge_upsert(target, source, key):
    """
    MERGE: update existing records, insert new ones.
    
    In BigQuery SQL, this is:
        MERGE INTO silver.calls AS target
        USING staging.incoming AS source
        ON target.call_id = source.call_id
        WHEN MATCHED THEN UPDATE SET ...
        WHEN NOT MATCHED THEN INSERT ...
    """
    # Find which incoming records already exist (MATCHED)
    existing_keys = set(target[key])
    updates = source[source[key].isin(existing_keys)]
    inserts = source[~source[key].isin(existing_keys)]
    
    # Apply updates: remove old version, add new version
    target = target[~target[key].isin(updates[key])]
    target = pd.concat([target, updates, inserts], ignore_index=True)
    
    print(f"MERGE result: {len(updates)} updated, {len(inserts)} inserted")
    return target

warehouse = merge_upsert(warehouse, incoming, "call_id")
print(f"\nWarehouse after MERGE: {len(warehouse)} records (was 10, now 11 — 1 new, 2 updated, no duplicates)")

In [ ]:
# Verify: C-006 and C-010 should be 'resolved' now, C-012 should be new
print("Verify the MERGE:")
warehouse[warehouse["call_id"].isin(["C-006", "C-010", "C-012"])][["call_id", "status", "duration"]]

**You Should See:** C-006 = resolved/480, C-010 = resolved/320, C-012 = in-progress/0. No duplicates.

---

## 7. Pattern: Dead Letter Queue (DLQ)

What happens when incoming data is bad? A null `call_id`, a negative `duration`, an impossible timestamp?

**Bad approach:** Drop the record silently. You will never know it was missing.

**Good approach:** Route it to a quarantine table (DLQ). Investigate later.

In [ ]:
# Simulate incoming data with some bad records
incoming_with_bad = pd.DataFrame({
    "call_id": ["C-020", None, "C-022", "C-023", "C-024"],
    "customer_id": ["CUST-120", "CUST-121", "CUST-122", "CUST-123", "CUST-124"],
    "status": ["resolved", "resolved", "INVALID_STATUS", "resolved", "resolved"],
    "duration": [200, 300, 150, -50, 100],
    "campaign_id": ["CAMP-A", "CAMP-B", "CAMP-A", "CAMP-C", "CAMP-B"],
    "created_at": pd.date_range("2026-04-14 09:00", periods=5, freq="30min"),
    "updated_at": pd.date_range("2026-04-14 09:00", periods=5, freq="30min"),
})

print("Incoming data (3 good, 3 bad):")
incoming_with_bad

In [ ]:
VALID_STATUSES = ["in-progress", "resolved", "missed", "voicemail", "transferred"]

def validate_and_route(df):
    """
    Validate records. Good records go to Silver. Bad records go to DLQ.
    Never silently drop data.
    """
    errors = []
    for idx, row in df.iterrows():
        row_errors = []
        
        # Check 1: call_id must not be null
        if pd.isna(row["call_id"]):
            row_errors.append("call_id is null")
        
        # Check 2: status must be valid
        if row["status"] not in VALID_STATUSES:
            row_errors.append(f"invalid status: {row['status']}")
        
        # Check 3: duration must be non-negative
        if row["duration"] < 0:
            row_errors.append(f"negative duration: {row['duration']}")
        
        errors.append("; ".join(row_errors) if row_errors else None)
    
    df = df.copy()
    df["validation_error"] = errors
    
    good = df[df["validation_error"].isna()].drop(columns=["validation_error"])
    bad = df[df["validation_error"].notna()]
    
    return good, bad

good_records, dlq_records = validate_and_route(incoming_with_bad)

print(f"Good records (→ Silver): {len(good_records)}")
print(f"Bad records (→ DLQ):     {len(dlq_records)}")

In [ ]:
# The DLQ shows WHAT failed and WHY
print("Dead Letter Queue:")
dlq_records[["call_id", "status", "duration", "validation_error"]]

**You Should See:** 3 bad records in the DLQ, each with a specific reason:
- null call_id
- invalid status
- negative duration

**The golden rule: Never silently drop data.** Quarantine it, log the reason, alert the team.

---

## 8. Idempotency — Safe to Re-run

An idempotent pipeline produces the same result whether you run it once or five times. No duplicates.

In [ ]:
# WHY: Pipelines fail and get retried. Airflow retries tasks automatically.
# If a retry creates duplicates, your data is wrong.

# Start with 11 records
warehouse = source_data.copy()
print(f"Before: {len(warehouse)} records")

# Run MERGE 3 times with the same incoming data
for run in range(1, 4):
    warehouse = merge_upsert(warehouse, incoming, "call_id")
    print(f"  After run {run}: {len(warehouse)} records")

print(f"\nAll 3 runs produced the same result: {len(warehouse)} records. No duplicates.")
print("This is idempotency. MERGE gives it to you automatically.")

In [ ]:
# ANTI-PATTERN: what happens with INSERT (append) instead of MERGE?
warehouse_bad = source_data.copy()
print(f"Before: {len(warehouse_bad)} records")

for run in range(1, 4):
    # INSERT (append) — NOT idempotent
    warehouse_bad = pd.concat([warehouse_bad, incoming], ignore_index=True)
    print(f"  After run {run}: {len(warehouse_bad)} records")

# Check for duplicates
dupes = warehouse_bad[warehouse_bad.duplicated(subset=["call_id"], keep=False)]
print(f"\nDuplicate records: {len(dupes)}")
print("This is why INSERT (append) without dedup is dangerous.")

**You Should See:** MERGE = same result every time. INSERT = duplicates pile up with every re-run.

---

## 9. The Complete Pattern

Putting it all together: incremental load + validation + DLQ + MERGE.

In [ ]:
def run_pipeline(source, warehouse, watermarks, table_name):
    """
    Complete pipeline: incremental + validate + DLQ + MERGE.
    
    In production (BigQuery), this is:
    1. SELECT * FROM bronze.calls WHERE updated_at > @watermark
    2. Validate and route bad records to pipeline.dlq
    3. MERGE INTO silver.calls USING staging ON call_id
    4. UPDATE pipeline.watermarks SET last_loaded_at = ...
    """
    print(f"--- Pipeline Run: {table_name} ---")
    
    # Step 1: Incremental extract
    watermark = watermarks.get(table_name, pd.Timestamp("1970-01-01"))
    new_records = source[source["updated_at"] > watermark]
    print(f"1. Extracted {len(new_records)} records since {watermark}")
    
    if len(new_records) == 0:
        print("   No new records. Done.")
        return warehouse, []
    
    # Step 2: Validate
    good, bad = validate_and_route(new_records)
    print(f"2. Validated: {len(good)} good, {len(bad)} → DLQ")
    
    # Step 3: MERGE good records
    if len(good) > 0:
        warehouse = merge_upsert(warehouse, good, "call_id")
    
    # Step 4: Update watermark
    watermarks[table_name] = new_records["updated_at"].max()
    print(f"4. Watermark → {watermarks[table_name]}")
    print(f"   Warehouse: {len(warehouse)} records")
    print()
    
    return warehouse, bad

# Reset
warehouse = source_data.copy()
watermarks = {}
all_dlq = []

# Run the pipeline
warehouse, dlq = run_pipeline(source_data, warehouse, watermarks, "calls")
if len(dlq) > 0:
    all_dlq.append(dlq)

**You Should See:** A complete pipeline run showing extract → validate → MERGE → watermark update.

---

## Summary

| Pattern | What It Does | When to Use |
|---|---|---|
| **ETL** | Transform outside warehouse | Heavy processing, multi-system joins, ML features |
| **ELT** | Transform inside warehouse (SQL) | BigQuery/Snowflake can handle it (most cases today) |
| **Full Refresh** | Drop and reload everything | Small tables, reference data |
| **Incremental** | Load only new records (watermark) | Growing tables, append-heavy |
| **MERGE** | Insert new + update existing | Records that change state (calls, orders) |
| **DLQ** | Quarantine bad records | Every production pipeline |
| **Idempotency** | Safe to re-run | Every production pipeline |

---

## Next Steps

- [ETL/ELT Patterns - How It Works](../../playbooks/data/pipelines/etl-elt/04_How_It_Works.md) — CDC mechanics, watermark internals
- [ETL/ELT Patterns - System Design](../../playbooks/data/pipelines/etl-elt/07_System_Design.md) — GCP and AWS architecture diagrams
- [ETL/ELT Patterns - Decision Guide](../../playbooks/data/pipelines/etl-elt/10_Decision_Guide.md) — Which pattern for which situation
- [Delta Lake Hello World](Delta_Lake_Hello_World.ipynb) — ACID transactions, MERGE on files, time travel

---

[Community](https://www.skool.com/deliverymomentum) | [Book a 1:1](https://calendly.com/sunil-mogadati/connect)